# 01 – Data Cleaning: Diabetes Hospital Readmissions

**Dataset:** UCI *Diabetes 130-US Hospitals for Years 1999–2008*, with about 100,000 hospital stays (called *encounters*) of patients with diabetes.

**Goal of this notebook:** turn the raw file into a clean, analysis-ready table and write down **why** each cleaning decision was made.

**Output:** `data/processed/cleaned_data.csv`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# This notebook lives in notebooks/, so the project folder is one level up.
# Adding it to sys.path lets us import our own code from src/.
PROJECT_DIR = Path.cwd().parent
sys.path.append(str(PROJECT_DIR))

from src.data_loader import (
    download_diabetes_data,
    download_id_mapping,
    load_id_mappings,
    load_raw_data,
)

# Show all columns when printing wide tables
pd.set_option("display.max_columns", 60)

## 1. Load the data

`download_diabetes_data()` and `download_id_mapping()` only download if the files are not already in `data/raw/`, so this cell is fast after the first run.

In [2]:
data_path = download_diabetes_data()
mapping_path = download_id_mapping()

df = load_raw_data(data_path)

# Keep the starting size so we can report it in the summary at the end
rows_raw, cols_raw = df.shape

File already exists, skipping download: C:\Users\ADITYA PANDEY\Downloads\data science\data science\01-healthcare-data-analysis\data\raw\diabetic_data.csv
File already exists, skipping download: C:\Users\ADITYA PANDEY\Downloads\data science\data science\01-healthcare-data-analysis\data\raw\IDS_mapping.csv


## 2. First look at the data

In [3]:
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")

Rows: 101,766   Columns: 50

In [4]:
# Data type of every column. 'object'/'str' columns hold text.
df.dtypes

encounter_id                int64
patient_nbr                 int64
race                          str
gender                        str
age                           str
weight                        str
admission_type_id           int64
discharge_disposition_id    int64
admission_source_id         int64
time_in_hospital            int64
payer_code                    str
medical_specialty             str
num_lab_procedures          int64
num_procedures              int64
num_medications             int64
number_outpatient           int64
number_emergency            int64
number_inpatient            int64
diag_1                        str
diag_2                        str
diag_3                        str
number_diagnoses            int64
max_glu_serum                 str
A1Cresult                     str
metformin                     str
repaglinide                   str
nateglinide                   str
chlorpropamide                str
glimepiride                   str
acetohexamide 

In [5]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


### Missing values are hidden as `"?"`

In the official dataset file, missing cells are not left empty: they contain a question mark `"?"`. pandas treats `"?"` as normal text, so a standard missing-value check would wrongly report **0 missing**. The fix is to convert every `"?"` to a real missing value (`NaN`).

**What I found:** our copy came from the `ucimlrepo` package, which **already converted** the `"?"` values to empty cells when it downloaded the data. So the count below is 0, and pandas already sees those cells as missing. We still run the conversion: it does no harm here, and it keeps this notebook correct if someone uses the original CSV from the UCI website or Kaggle instead.

In [6]:
# How many "?" are in each column (before converting)
question_marks = (df == "?").sum()
question_marks[question_marks > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [7]:
# Replace every "?" with NaN (pandas' marker for "missing")
df = df.replace("?", np.nan)


def missing_table(data):
    """Return a table of columns that have missing values, worst first."""
    table = pd.DataFrame({
        "missing_count": data.isna().sum(),
        "missing_percent": (data.isna().mean() * 100).round(2),
    })
    table = table[table["missing_count"] > 0]
    return table.sort_values("missing_percent", ascending=False)


missing_table(df)

,missing_count,missing_percent
weight,98569,96.86
max_glu_serum,96420,94.75
A1Cresult,84748,83.28
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


## 3. Handle missing values, column by column

The rule I followed:
- **Drop a column** when most of it is missing **and** it can't be reliably filled.
- **Label as "Unknown" / "Not tested"** when the column is useful and the fact that it is missing tells us something.
- **Drop rows** only when very few rows are affected.

| Column | Missing | Decision |
|---|---|---|
| `weight` | ~97% | Drop column |
| `payer_code` | ~40% | Drop column |
| `max_glu_serum`, `A1Cresult` | ~95% / ~83% | Label as "Not tested" |
| `medical_specialty` | ~49% | Label as "Unknown" |
| `race` | ~2% | Label as "Unknown" |
| `diag_1`, `diag_2`, `diag_3` | <1.5% | Label as "Unknown" |

### 3.1 `weight`: drop the column

About **97%** of values are missing. With only ~3% filled in, any number we used to fill the gaps (like the average) would be a guess for almost every patient, and the column would mislead more than it helps. The original study behind this dataset (Strack et al., 2014) also dropped `weight` for this reason.

In [8]:
df = df.drop(columns=["weight"])

### 3.2 `payer_code`: drop the column

`payer_code` is the patient's insurance type (for example Medicare). About **40%** is missing. Insurance type is not part of the clinical question this project studies (which patients come back to hospital within 30 days), and filling 40% with "Unknown" would make "Unknown" one of the biggest groups. The original study dropped it as well.

In [9]:
df = df.drop(columns=["payer_code"])

### 3.3 `max_glu_serum` and `A1Cresult`: label as "Not tested"

These are lab test results (blood glucose and HbA1c). A missing value here does **not** mean the data was lost; it means **the test was not done** during that stay. That is useful information: the original study found that whether HbA1c was measured is linked to readmission. So we keep the columns and fill the gaps with the label `"Not tested"`.

(In the official CSV these cells contain the word `"None"`; `ucimlrepo` turned them into empty cells. The code handles both forms.)

In [10]:
for col in ["max_glu_serum", "A1Cresult"]:
    df[col] = df[col].fillna("Not tested")             # empty cells (ucimlrepo version)
    df[col] = df[col].replace("None", "Not tested")    # "None" text (original CSV)

df["A1Cresult"].value_counts()

A1Cresult
Not tested    84748
>8             8216
Norm           4990
>7             3812
Name: count, dtype: int64

### 3.4 `medical_specialty`: label as "Unknown"

About **49%** is missing, but this column says which kind of doctor admitted the patient (Cardiology, Surgery and so on), which can be useful to compare readmission rates. Dropping half the rows would throw away too much data, so we keep the column and label missing values `"Unknown"`. This column also has many tiny categories; we can group them later during analysis.

In [11]:
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")

### 3.5 `race`: label as "Unknown"

Only about **2%** is missing. Dropping those rows would be acceptable, but labelling them `"Unknown"` keeps every patient and stays honest that we don't know their race. Guessing the most common race would be wrong and could bias the results.

In [12]:
df["race"] = df["race"].fillna("Unknown")

### 3.6 `diag_1`, `diag_2`, `diag_3`: label as "Unknown"

These are the primary, secondary and additional diagnosis codes. Very few are missing (under 1.5%). Many patients only have one or two diagnoses recorded, so a missing secondary diagnosis is normal. We label the gaps `"Unknown"` and keep the rows.

In [13]:
for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].fillna("Unknown")

### 3.7 `gender`: remove 3 "Unknown/Invalid" rows

`gender` has no `"?"`, but 3 rows say `"Unknown/Invalid"`, which is another kind of hidden missing value. Three rows out of 100,000 is too few to matter, so we simply remove them.

In [14]:
print(df["gender"].value_counts())

df = df[df["gender"] != "Unknown/Invalid"]
rows_after_gender = len(df)

gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64


### 3.8 Drop columns with only one value

`examide` and `citoglipton` (two diabetes medicines) are `"No"` for **every** patient. A column where every value is the same can't tell patients apart, so it adds nothing to the analysis.

In [15]:
# nunique() counts the different values in each column
single_value_cols = [col for col in df.columns if df[col].nunique() == 1]
print("Columns with only one value:", single_value_cols)

df = df.drop(columns=single_value_cols)

Columns with only one value: ['examide', 'citoglipton']


### Check: no missing values left

In [16]:
missing_table(df)  # an empty table means nothing is missing

,missing_count,missing_percent


## 4. Remove duplicates

First, check for **exact duplicate rows** (every column identical) and repeated `encounter_id`s. Both should be zero, because each row is one hospital stay.

In [17]:
print("Exact duplicate rows:  ", df.duplicated().sum())
print("Duplicate encounter_id:", df["encounter_id"].duplicated().sum())

Exact duplicate rows:   0
Duplicate encounter_id: 0


### Keep one encounter per patient

Many patients appear more than once because they were admitted several times. If we keep every stay:
- **Frequent patients are counted many times**, so a few very sick people can pull the averages and rates their way.
- **Rows are no longer independent.** Statistics and models assume each row is a separate person; repeated stays of the same person break that.

So we keep **one encounter per patient** (`patient_nbr`): their **first** one. We sort by `encounter_id` and assume that a smaller ID means an earlier visit. The original study made the same choice. The `readmitted` column still tells us whether that first stay led to a readmission.

In [18]:
print(f"Encounters: {len(df):,}")
print(f"Unique patients: {df['patient_nbr'].nunique():,}")

df = (
    df.sort_values("encounter_id")                       # earliest visit first
      .drop_duplicates(subset="patient_nbr", keep="first")
)

rows_after_dedup = len(df)
print(f"Rows after keeping one encounter per patient: {rows_after_dedup:,}")

Encounters: 101,763
Unique patients: 71,515


Rows after keeping one encounter per patient: 71,515


## 5. Map ID columns to readable labels

`admission_type_id`, `discharge_disposition_id` and `admission_source_id` are numbers (for example `1`, `7`). On their own they mean nothing in a chart. The official `IDS_mapping.csv` file from UCI explains each number; for example `admission_type_id = 1` means *Emergency*. `load_id_mappings()` reads that file into Python dictionaries.

In [19]:
id_mappings = load_id_mappings(mapping_path)

# Example: the admission type table
id_mappings["admission_type_id"]

{1: 'Emergency',
 2: 'Urgent',
 3: 'Elective',
 4: 'Newborn',
 5: 'Not Available',
 6: 'NULL',
 7: 'Trauma Center',
 8: 'Not Mapped'}

Several codes all mean "we don't know": `NULL`, `Not Available`, `Not Mapped` and `Unknown/Invalid`. Keeping four separate "unknown" groups would split the same idea across four bars in every chart, so we merge them into one label, `"Unknown"`.

We create new `*_label` columns and keep the original IDs, so we can always check the mapping.

In [20]:
unknown_labels = ["NULL", "Not Available", "Not Mapped", "Unknown/Invalid"]

for id_col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    label_col = id_col.replace("_id", "_label")   # e.g. admission_type_label
    df[label_col] = df[id_col].map(id_mappings[id_col])
    df[label_col] = df[label_col].replace(unknown_labels, "Unknown")

# Every ID should have found a label
print("IDs with no label:", df[["admission_type_label",
                                "discharge_disposition_label",
                                "admission_source_label"]].isna().sum().sum())

df["admission_type_label"].value_counts()

IDs with no label: 0


admission_type_label
Emergency        36488
Elective         13916
Urgent           13028
Unknown           8053
Trauma Center       21
Newborn              9
Name: count, dtype: int64

### Remove patients who could not be readmitted

Some stays ended with the patient **passing away** or going to **hospice** (end-of-life care). These patients can't come back to hospital, so they would always be counted as "not readmitted" and make the readmission rate look better than it really is. We remove any stay whose discharge label contains *Expired* or *Hospice*, as the original study did.

In [21]:
cannot_return = df["discharge_disposition_label"].str.contains("Expired|Hospice")
print("Stays ending in death or hospice:", cannot_return.sum())

df = df[~cannot_return]   # ~ means "not"
rows_after_expired = len(df)

Stays ending in death or hospice: 1545


## 6. Create clean analysis columns

| New column | Built from | Why |
|---|---|---|
| `age_group` | `age` (like `"[70-80)"`) | Easier to read: `"70-79"` |
| `readmitted_30days` | `readmitted` | Yes/No answer to the main question: back within 30 days? |
| `length_of_stay` | `time_in_hospital` | Clearer name; the value is already the number of days in hospital (1–14) |

In [22]:
# age: "[70-80)" -> take the lower number (70) -> "70-79"
age_lower = df["age"].str.extract(r"(\d+)-")[0].astype(int)
df["age_group"] = age_lower.astype(str) + "-" + (age_lower + 9).astype(str)

df[["age", "age_group"]].drop_duplicates().sort_values("age")

,age,age_group
0,[0-10),0-9
1,[10-20),10-19
2,[20-30),20-29
3,[30-40),30-39
4,[40-50),40-49
5,[50-60),50-59
11,[60-70),60-69
7,[70-80),70-79
8,[80-90),80-89
9,[90-100),90-99


`readmitted` has three values: `"<30"` (back within 30 days), `">30"` (back after more than 30 days) and `"NO"`. Hospitals are measured on **30-day** readmissions (for example by the US Hospital Readmissions Reduction Program), so the main target is **Yes** only for `"<30"`. We keep the original `readmitted` column too, for more detailed analysis later.

In [23]:
df["readmitted_30days"] = np.where(df["readmitted"] == "<30", "Yes", "No")

df["readmitted_30days"].value_counts(normalize=True).round(3)

readmitted_30days
No     0.91
Yes    0.09
Name: proportion, dtype: float64

In [24]:
# time_in_hospital is already a whole number of days, so we just give it a clearer name
df = df.rename(columns={"time_in_hospital": "length_of_stay"})

# The old age text is replaced by age_group, so we drop it
df = df.drop(columns=["age"])

df["length_of_stay"].describe()

count    69970.000000
mean         4.273260
std          2.933901
min          1.000000
25%          2.000000
50%          3.000000
75%          6.000000
max         14.000000
Name: length_of_stay, dtype: float64

## 7. Save the cleaned data

In [25]:
output_path = PROJECT_DIR / "data" / "processed" / "cleaned_data.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print(f"Saved {len(df):,} rows and {df.shape[1]} columns to {output_path}")

Saved 69,970 rows and 50 columns to C:\Users\ADITYA PANDEY\Downloads\data science\data science\01-healthcare-data-analysis\data\processed\cleaned_data.csv


In [26]:
# Numbers for the summary below
print(f"Rows before cleaning:            {rows_raw:,}")
print(f"After removing invalid gender:   {rows_after_gender:,}  (-{rows_raw - rows_after_gender:,})")
print(f"After one encounter per patient: {rows_after_dedup:,}  (-{rows_after_gender - rows_after_dedup:,})")
print(f"After removing death/hospice:    {rows_after_expired:,}  (-{rows_after_dedup - rows_after_expired:,})")
print(f"Rows kept: {rows_after_expired / rows_raw:.1%} of the original")
print(f"Columns: {cols_raw} before, {df.shape[1]} after")

Rows before cleaning:            101,766
After removing invalid gender:   101,763  (-3)
After one encounter per patient: 71,515  (-30,248)
After removing death/hospice:    69,970  (-1,545)
Rows kept: 68.8% of the original
Columns: 50 before, 50 after


## Cleaning summary

### Rows: 101,766 → 69,970 (68.8% kept)

| Step | Rows removed | Rows left | Why |
|---|---:|---:|---|
| Raw data | – | 101,766 | One row per hospital stay |
| Removed invalid gender | 3 | 101,763 | `"Unknown/Invalid"` is a hidden missing value; too few rows to matter |
| Kept first encounter per patient | 30,248 | 71,515 | Avoids counting frequent patients many times; keeps rows independent |
| Removed death / hospice discharges | 1,545 | 69,970 | These patients could not be readmitted, so they would make the readmission rate look lower than it is |

No exact duplicate rows or repeated `encounter_id`s were found.

### Columns: 50 → 50 (5 removed, 5 added, 1 renamed)

| Column | Action | Why |
|---|---|---|
| `weight` | Dropped | ~97% missing; too little data to fill reliably |
| `payer_code` | Dropped | ~40% missing; insurance type is not part of the clinical question |
| `examide`, `citoglipton` | Dropped | The same value (`"No"`) for every patient, so they can't tell patients apart |
| `age` | Replaced by `age_group` | `"[70-80)"` becomes the easier-to-read `"70-79"` |
| `time_in_hospital` | Renamed to `length_of_stay` | Clearer name (days in hospital, 1–14) |
| `admission_type_label`, `discharge_disposition_label`, `admission_source_label` | Added | Readable labels from the official `IDS_mapping.csv`; "unknown"-type codes merged into `"Unknown"` |
| `readmitted_30days` | Added | Yes/No target: readmitted within 30 days |

### Missing values: labelled, not guessed
- `max_glu_serum`, `A1Cresult`: `"Not tested"`, because missing means the test was not done
- `medical_specialty`, `race`, `diag_1`–`diag_3`: `"Unknown"`

The cleaned data has **no missing values**.

### Result
The 30-day readmission rate in the cleaned data is about **9%**, measured on one first stay per patient. It is lower than in the raw data because repeat visits were removed, and frequent visitors are more likely to be readmitted.